[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/27_deep_learning_foundations/02_computational_graph_and_backprop.ipynb)

# Bài 02 — Đồ thị tính toán và lan truyền ngược

**🎯 Mục tiêu**
- Biểu diễn phép tính bằng **đồ thị tính toán**; quy tắc chuỗi; **gradient cục bộ**.
- Lan truyền ngược (backprop) tính tay trên một ví dụ vô hướng; **kiểm tra gradient** bằng sai phân hữu hạn.
- Tự xây một **engine autograd** nhỏ (kiểu micrograd): mỗi phép toán biết đạo hàm cục bộ của nó, `backward()` duyệt đồ thị theo thứ tự tô-pô.
- Huấn luyện một mạng nơ-ron nhỏ bằng engine đó; đối chiếu với `torch.autograd`.
- Backprop dạng ma trận cho lớp tuyến tính; vì sao backprop hiệu quả (so với sai phân hữu hạn).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/27_deep_learning_foundations

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import make_classification_2d
from ml_course.visualization import set_style

set_style()
try:
    import torch
    HAS_TORCH = True
except ImportError:                                                     # PyTorch là tuỳ chọn ở bài này: pip install -e ".[dl]"
    HAS_TORCH = False

## 1. Đồ thị tính toán và quy tắc chuỗi

Xét $f(x, y, z, w) = (x\,y + z)\cdot\sigma(w)$. Tách thành các bước nhỏ, mỗi bước là một **nút**:

```
x ─┐
   (×) → q = x·y ─┐
y ─┘              (+) → s = q + z ─┐
z ────────────────┘                (×) → f = s · r
w → (σ) → r = σ(w) ────────────────┘
```

- **Forward**: tính giá trị từ trái sang phải.
- **Backward**: đi từ $f$ ngược về, nhân **gradient từ phía sau** với **gradient cục bộ** của từng nút (quy tắc chuỗi):
$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial s}\cdot\frac{\partial s}{\partial q}\cdot\frac{\partial q}{\partial x} = r \cdot 1 \cdot y$$

In [ ]:
x, y, z, w = 2.0, -3.0, 4.0, 0.5
# forward
q = x * y
s = q + z
r = 1 / (1 + math.exp(-w))
f = s * r
# backward: mỗi dòng = gradient phía sau × gradient cục bộ
df_df = 1.0
df_ds = r * df_df
df_dr = s * df_df
df_dq = 1.0 * df_ds                                                     # nút cộng: chuyển gradient nguyên vẹn
df_dz = 1.0 * df_ds
df_dx = y * df_dq                                                       # nút nhân: gradient = giá trị nhánh kia
df_dy = x * df_dq
df_dw = r * (1 - r) * df_dr                                             # σ'(w) = σ(w)(1 − σ(w))
print(f"f = {f:.4f}")
print(f"∂f/∂x = {df_dx:.4f}, ∂f/∂y = {df_dy:.4f}, ∂f/∂z = {df_dz:.4f}, ∂f/∂w = {df_dw:.4f}")

Các "mẫu" gradient cần nhớ:
- **Nút cộng** phân phối gradient nguyên vẹn cho mọi đầu vào.
- **Nút nhân** gửi cho mỗi đầu vào gradient × giá trị của đầu vào **kia** (đổi chỗ).
- **Nút max** (ReLU) chỉ chuyển gradient cho đầu vào lớn nhất.
- Một biến dùng ở **nhiều** nơi → **cộng** các gradient từ mọi nhánh.

### Kiểm tra gradient bằng sai phân hữu hạn

$$\frac{\partial f}{\partial x} \approx \frac{f(x + h) - f(x - h)}{2h}$$
— luôn dùng để kiểm tra code backprop tự viết.

In [ ]:
def f_func(x, y, z, w):
    return (x * y + z) / (1 + math.exp(-w))


h = 1e-5
numeric = [(f_func(*[v + h * (k == j) for k, v in enumerate([x, y, z, w])]) -
            f_func(*[v - h * (k == j) for k, v in enumerate([x, y, z, w])])) / (2 * h) for j in range(4)]
analytic = [df_dx, df_dy, df_dz, df_dw]
print("sai phân hữu hạn:", np.round(numeric, 6))
print("backprop        :", np.round(analytic, 6))
print(f"sai lệch tương đối lớn nhất: {np.max(np.abs(np.array(numeric) - analytic) / (np.abs(analytic) + 1e-12)):.2e}")

## 2. Một engine autograd nhỏ

Ý tưởng (theo *micrograd* của Andrej Karpathy): mỗi số là một đối tượng `Value` ghi nhớ (1) giá trị, (2) các nút cha tạo ra nó, (3) cách đẩy gradient về cha. `backward()` sắp xếp đồ thị theo **thứ tự tô-pô** rồi gọi các hàm đẩy gradient từ cuối về đầu.

In [ ]:
class Value:
    def __init__(self, data, parents=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self._parents = parents
        self._backward = lambda: None
        self.op = op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad                                       # += : biến có thể được dùng nhiều lần
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data**k, (self,), f"**{k}")
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")
        def _backward():
            self.grad += (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), "exp")
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(math.log(self.data), (self,), "log")
        def _backward():
            self.grad += out.grad / self.data
        out._backward = _backward
        return out

    def backward(self):
        order, visited = [], set()
        def build(v):                                                   # thứ tự tô-pô: cha trước con
            if id(v) not in visited:
                visited.add(id(v))
                for p in v._parents:
                    build(p)
                order.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

    # các phép suy ra từ phép cơ bản
    __radd__ = __add__
    __rmul__ = __mul__
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return (-self) + other
    def __truediv__(self, other): return self * (other ** -1 if isinstance(other, Value) else 1.0 / other)
    def sigmoid(self): return (1 + (-self).exp()) ** -1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

In [ ]:
xv, yv, zv, wv = Value(x), Value(y), Value(z), Value(w)
fv = (xv * yv + zv) * wv.sigmoid()
fv.backward()
print("f =", round(fv.data, 4))
print("grad engine :", [round(v.grad, 6) for v in (xv, yv, zv, wv)])
print("grad tay    :", [round(g, 6) for g in analytic])

Vẽ đồ thị mà engine vừa xây cho $f$ (đầu vào bên trái, đầu ra bên phải). Mỗi nút ghi **phép toán**, **giá trị** (forward) và **gradient** $\partial f/\partial(\text{nút})$ (backward):

In [ ]:
def draw_graph(root, names=None):
    """Vẽ đồ thị tính toán của một Value bằng matplotlib."""
    names = names or {}
    level, nodes, edges, stack = {id(root): 0}, {id(root): root}, set(), [root]
    while stack:                                                        # mức của nút = đường dài nhất tới đầu ra
        v = stack.pop()
        for p in v._parents:
            edges.add((id(p), id(v)))
            if level.get(id(p), -1) < level[id(v)] + 1:
                level[id(p)], nodes[id(p)] = level[id(v)] + 1, p
                stack.append(p)
    depth = max(level.values())
    columns = {}
    for key, lv in level.items():
        columns.setdefault(lv, []).append(key)
    pos = {key: ((depth - lv) * 1.8, (i - (len(keys) - 1) / 2) * 1.4)
           for lv, keys in columns.items() for i, key in enumerate(keys)}
    fig, ax = plt.subplots(figsize=(1.9 * (depth + 1), 1.5 * max(len(k) for k in columns.values()) + 0.8))
    for a, b in edges:
        ax.annotate("", xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle="-|>", color="0.55", shrinkA=24, shrinkB=24))
    for key, v in nodes.items():
        if key == id(root):
            color, title = "#c8e6c9", f"đầu ra ({v.op})"
        elif not v._parents:
            color, title = ("#bbdefb", names[key]) if key in names else ("#eeeeee", "hằng số")
        else:
            color, title = "#fff3c4", v.op
        ax.text(*pos[key], f"{title}\n{v.data:.3f}\n∂f = {v.grad:.3f}", ha="center", va="center", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.4", fc=color, ec="0.4"))
    ax.set_xlim(-1.1, depth * 1.8 + 1.1)
    ax.set_ylim(min(y for _, y in pos.values()) - 0.9, max(y for _, y in pos.values()) + 0.9)
    ax.axis("off")
    ax.set_title("Đồ thị tính toán của f = (x·y + z)·σ(w): giá trị forward và gradient backward")
    plt.show()


draw_graph(fv, names={id(xv): "x", id(yv): "y", id(zv): "z", id(wv): "w"})

Đây chính là sơ đồ ở mục 1, nhưng $\sigma(w)$ được **ghép từ các phép cơ bản** (nhân với −1, `exp`, cộng 1, mũ −1) — engine chỉ cần biết đạo hàm của vài phép cơ bản là tính được gradient của mọi biểu thức ghép từ chúng. Gradient ở các nút đầu vào (xanh dương) khớp bảng tính tay ở trên.

Engine cho đúng các gradient đã tính tay. Một biến được dùng **nhiều lần** (vd $a \cdot a + a$) phải cộng dồn gradient — lý do dùng `+=`:

In [ ]:
a = Value(3.0)
g = a * a + a                                                           # dg/da = 2a + 1 = 7
g.backward()
print(f"g = a·a + a tại a = 3: dg/da = {a.grad} (đúng: 7)")

## 3. Huấn luyện mạng nơ-ron bằng engine tự viết

Mạng 2 → 8 → 8 → 1 (tanh), loss **binary cross-entropy**, gradient descent toàn bộ batch trên 60 điểm "moons". Mỗi bước: forward xây đồ thị, `backward()`, cập nhật, **đặt lại gradient về 0**.

In [ ]:
class Neuron:
    def __init__(self, n_in, r, act=True):
        self.w = [Value(r.normal(0, 1 / math.sqrt(n_in))) for _ in range(n_in)]
        self.b = Value(0.0)
        self.act = act

    def __call__(self, xs):
        z = sum((wi * xi for wi, xi in zip(self.w, xs)), self.b)
        return z.tanh() if self.act else z

    def parameters(self):
        return self.w + [self.b]


class MLP:
    def __init__(self, sizes, seed=0):
        r = np.random.default_rng(seed)
        self.layers = [[Neuron(sizes[i], r, act=(i < len(sizes) - 2)) for _ in range(sizes[i + 1])] for i in range(len(sizes) - 1)]

    def __call__(self, xs):
        for layer in self.layers:
            xs = [n(xs) for n in layer]
        return xs[0]

    def parameters(self):
        return [p for layer in self.layers for n in layer for p in n.parameters()]


X_m, y_m = make_classification_2d("moons", n=60, noise=0.1, seed=0)
net = MLP([2, 8, 8, 1])
params = net.parameters()
print(f"số tham số: {len(params)}")
history = []
t0 = time.perf_counter()
for step in range(60):
    logits = [net([Value(a), Value(b)]) for a, b in X_m]
    # BCE viết theo logit z: −[y·log σ(z) + (1 − y)·log(1 − σ(z))] = log(1 + e^z) − y·z
    loss = sum(((lg.exp() + 1).log() - yi * lg for lg, yi in zip(logits, y_m)), Value(0.0)) * (1 / len(X_m))
    for p in params:
        p.grad = 0.0                                                    # quên bước này → gradient cộng dồn qua các bước
    loss.backward()
    for p in params:
        p.data -= 0.5 * p.grad
    acc = np.mean([(lg.data > 0) == yi for lg, yi in zip(logits, y_m)])
    history.append((loss.data, acc))
print(f"60 bước trong {time.perf_counter() - t0:.1f}s | loss {history[0][0]:.3f} → {history[-1][0]:.3f} | accuracy {history[0][1]:.2f} → {history[-1][1]:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot([h[0] for h in history])
axes[0].set_title("loss (BCE)")
axes[0].set_xlabel("bước")
gx, gy = np.meshgrid(np.linspace(-1.5, 2.5, 50), np.linspace(-1, 1.5, 40))
Zg = np.array([net([Value(a), Value(b)]).data for a, b in zip(gx.ravel(), gy.ravel())]).reshape(gx.shape)
axes[1].contourf(gx, gy, Zg > 0, alpha=0.3, cmap="coolwarm")
axes[1].scatter(*X_m.T, c=y_m, cmap="coolwarm", edgecolors="k")
axes[1].set_title("ranh giới học được bằng engine tự viết")
plt.show()

Engine vô hướng này cực chậm (mỗi số là một đối tượng Python), nhưng nguyên lý giống hệt PyTorch/JAX: các thư viện thật làm việc trên **tensor** (mỗi nút là phép toán ma trận, chạy trên C++/GPU).

## 4. Đối chiếu với `torch.autograd`

In [ ]:
if HAS_TORCH:
    tx, ty, tz, tw = (torch.tensor(v, requires_grad=True, dtype=torch.float64) for v in (x, y, z, w))
    tf = (tx * ty + tz) * torch.sigmoid(tw)
    tf.backward()
    print("torch.autograd:", [round(t.grad.item(), 6) for t in (tx, ty, tz, tw)])
    print("engine tự viết:", [round(v.grad, 6) for v in (xv, yv, zv, wv)])
else:
    print("Bỏ qua — cài PyTorch để đối chiếu.")

## 5. Backprop dạng ma trận: lớp tuyến tính

Với $Z = XW + \mathbf b$ ($X$: $n \times d$, $W$: $d \times m$) và gradient từ phía sau $\Delta = \partial L/\partial Z$ ($n \times m$):
$$\frac{\partial L}{\partial W} = X^\top\Delta, \qquad \frac{\partial L}{\partial\mathbf b} = \sum_{\text{hàng}}\Delta, \qquad \frac{\partial L}{\partial X} = \Delta W^\top$$
Mẹo nhớ: **kích thước phải khớp** — $\partial L/\partial W$ cùng kích thước $W$ ($d \times m$) → chỉ có $X^\top\Delta$ khớp.

In [ ]:
rng = np.random.default_rng(0)
Xn, Wn, bn = rng.normal(size=(5, 3)), rng.normal(size=(3, 4)), rng.normal(size=4)
T = rng.normal(size=(5, 4))
loss_fn = lambda W: 0.5 * np.sum((Xn @ W + bn - T) ** 2)                # L = ½‖XW + b − T‖²
Delta = Xn @ Wn + bn - T                                                # ∂L/∂Z
dW_analytic = Xn.T @ Delta
dW_numeric = np.zeros_like(Wn)
for i in range(3):
    for j in range(4):
        E = np.zeros_like(Wn)
        E[i, j] = 1e-6
        dW_numeric[i, j] = (loss_fn(Wn + E) - loss_fn(Wn - E)) / 2e-6
print(f"‖dW giải tích − dW số‖ / ‖dW‖ = {np.linalg.norm(dW_analytic - dW_numeric) / np.linalg.norm(dW_analytic):.2e}")

## 6. Vì sao backprop hiệu quả?

Sai phân hữu hạn cần **2 lần forward cho mỗi tham số** → với $P$ tham số: $O(P)$ lần forward. Backprop tính **mọi** gradient với chi phí ≈ **một** lần forward + một lần backward (cỡ 2–3 lần forward) — bất kể $P$. Mạng có hàng tỉ tham số chỉ huấn luyện được nhờ điều này (reverse-mode automatic differentiation).

In [ ]:
for n_params in [10, 100, 1000]:
    Wt = rng.normal(size=(n_params, 1))
    Xt = rng.normal(size=(2000, n_params))
    loss_t = lambda W: np.mean((Xt @ W) ** 2)
    t0 = time.perf_counter()
    grad_bp = 2 * Xt.T @ (Xt @ Wt) / len(Xt)                            # backprop (giải tích)
    t_bp = time.perf_counter() - t0
    t0 = time.perf_counter()
    for k in range(n_params):                                           # sai phân: 2 forward mỗi tham số
        E = np.zeros_like(Wt)
        E[k] = 1e-6
        _ = (loss_t(Wt + E) - loss_t(Wt - E)) / 2e-6
    t_fd = time.perf_counter() - t0
    print(f"{n_params:5d} tham số: backprop {t_bp * 1000:7.2f} ms | sai phân hữu hạn {t_fd * 1000:9.1f} ms")

## 🧠 Kiểm tra nhanh

1. Nút cộng và nút nhân chuyển gradient về đầu vào thế nào?
2. Vì sao phải **cộng dồn** gradient khi một biến được dùng nhiều lần? Hệ quả trong vòng lặp huấn luyện?
3. Vì sao `backward()` cần thứ tự tô-pô?
4. Viết $\partial L/\partial W$, $\partial L/\partial X$ cho $Z = XW$.
5. Vì sao backprop rẻ hơn sai phân hữu hạn rất nhiều? Khi nào vẫn dùng sai phân hữu hạn?

## 📌 Tóm tắt

- Đồ thị tính toán + quy tắc chuỗi: gradient = gradient phía sau × gradient cục bộ, cộng dồn qua các nhánh.
- Autograd: mỗi phép toán biết đạo hàm cục bộ; backward duyệt ngược theo thứ tự tô-pô.
- Lớp tuyến tính: $X^\top\Delta$, $\sum\Delta$, $\Delta W^\top$; kiểm tra gradient bằng sai phân hữu hạn; backprop có chi phí cỡ một lần forward.

## 📚 Tài liệu tham khảo

- Rumelhart, Hinton & Williams (1986) — *Learning representations by back-propagating errors*
- Karpathy — *micrograd* (github.com/karpathy/micrograd) và bài giảng "The spelled-out intro to neural networks and backpropagation"
- CS231n — *Backpropagation, Intuitions* (cs231n.github.io/optimization-2)
- Baydin et al. (2018) — *Automatic Differentiation in Machine Learning: a Survey*